# X-ray preprocessing exploration

Interactive companion to `src/preprocess.py`. Use this to:
- Eyeball raw frames and spot which pages have real data
- Tune percentile / CLAHE / gamma parameters visually
- Compare individual frames vs. the mean-stacked frame
- Produce figures for Vineet

**Context:** Dry-ice sublimation X-ray imaging, proof-of-concept for ML frost-formation rendering from sparse imaging.

**Current limitations (flag to lab):**
- `dark.tiff` is all zeros → dark subtraction is a no-op
- No flat-field file → pipeline falls back to `raw` instead of `(raw - dark)/(flat - dark)`
- Only frames 0–6 of `object1_hole.tiff` have usable data; 7 is partial, 8+ are empty

In [ ]:
import sys
from pathlib import Path

# Make src/ importable
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import tifffile

from preprocess import (
    PreprocConfig, load_tiff_stack, select_usable_frames,
    flat_field_correct, percentile_normalize,
    apply_clahe, apply_gamma, process_frame,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load + inventory the stacks

In [ ]:
# Update these paths to wherever your data lives on the M4
OBJECT_PATH = '../../data/object1_hole.tiff'
DARK_PATH   = '../../data/dark.tiff'
FLAT_PATH   = None  # fill in when you have one

obj_stack = load_tiff_stack(OBJECT_PATH)
print(f'object stack: {obj_stack.shape}, dtype={obj_stack.dtype}')

# Per-page inventory -- quickly spot which frames are real
for i in range(len(obj_stack)):
    p = obj_stack[i]
    nz_pct = 100 * np.count_nonzero(p) / p.size
    tag = '  <-- real data' if nz_pct > 50 else ('  (partial)' if nz_pct > 1 else '  (empty)')
    print(f'  page {i:2d}: max={p.max():5d}, mean={p.mean():6.1f}, nonzero={nz_pct:5.1f}%{tag}')

In [ ]:
# Dark-frame sanity check
if DARK_PATH:
    dark_stack = load_tiff_stack(DARK_PATH)
    print(f'dark stack: {dark_stack.shape}')
    print(f'  all-zero? {bool((dark_stack == 0).all())}')
    print(f'  total non-zero pixels: {np.count_nonzero(dark_stack)} / {dark_stack.size}')
    # If dark is garbage, we want to KNOW so we can nag Saya's lab
else:
    dark_stack = None

## 2. Grab usable frames and build a mean stack

In [ ]:
cfg = PreprocConfig()   # defaults: pct=1/99, CLAHE clip=0.01, gamma=0.6

frames = select_usable_frames(obj_stack, *cfg.usable_frames)
mean_frame = frames.mean(axis=0)
print(f'usable frames: {frames.shape}')
print(f'mean frame:    min={mean_frame.min():.1f}, max={mean_frame.max():.1f}, mean={mean_frame.mean():.1f}')

# SNR sanity check: std of a flat-ish region. For the mean stack it should be
# ~sqrt(N) lower than a single frame.
roi = (slice(100, 200), slice(200, 400))  # adjust to pick a visually flat patch
print(f'\nROI std (single frame 0): {frames[0][roi].std():.2f}')
print(f'ROI std (mean of {len(frames)}): {mean_frame[roi].std():.2f}')
print(f'Expected ~sqrt({len(frames)}) = {np.sqrt(len(frames)):.2f}x reduction')

## 3. Tune parameters visually

Change `cfg` values and re-run. `clip_limit` is the main CLAHE knob — higher = more contrast but also more noise amplification.

In [ ]:
stages = process_frame(mean_frame, cfg, dark=dark_stack)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
panels = [
    ('Raw (auto-scaled)',    stages['raw']),
    ('Percentile-normalized', stages['normalized']),
    ('CLAHE',                stages['clahe']),
    ('Gamma',                stages['gamma']),
]
for ax, (name, img) in zip(axes.ravel(), panels):
    im = ax.imshow(img, cmap='gray')
    ax.set_title(name)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f'mean stack | pct=[{cfg.pct_low},{cfg.pct_high}] clip={cfg.clahe_clip_limit} gamma={cfg.gamma}')
fig.tight_layout();

In [ ]:
# Parameter sweep: try a few CLAHE clip limits side by side
norm = stages['normalized']
clips = [0.005, 0.01, 0.02, 0.04]

fig, axes = plt.subplots(1, len(clips), figsize=(5*len(clips), 5))
for ax, c in zip(axes, clips):
    out = apply_clahe(norm, clip_limit=c, kernel_divisor=8)
    ax.imshow(out, cmap='gray')
    ax.set_title(f'clip_limit={c}')
    ax.axis('off')
fig.tight_layout();

## 4. Per-frame vs mean-stack: does averaging help?

Should see cleaner noise floor in the mean-stacked normalized image.

In [ ]:
single = percentile_normalize(frames[0], cfg.pct_low, cfg.pct_high)
stacked = percentile_normalize(mean_frame, cfg.pct_low, cfg.pct_high)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(single, cmap='gray'); axes[0].set_title('Frame 0 only')
axes[1].imshow(stacked, cmap='gray'); axes[1].set_title(f'Mean of {len(frames)} frames')
for ax in axes: ax.axis('off')
fig.tight_layout();

## 5. Scratch space

Plots to debug specific issues, try new ideas, etc.